In [1]:
import pandas as pd
df_2021 = pd.read_csv("/content/world-happiness-report-2021.csv")
df_panel = pd.read_csv("/content/world-happiness-report.csv")


columns_to_drop = [
    "Standard error of ladder score",
    "upperwhisker",
    "lowerwhisker",
    "Ladder score in Dystopia",
    "Explained by: Log GDP per capita",
    "Explained by: Generosity",
    "Explained by: Social support",
    "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices",
    "Explained by: Perceptions of corruption",
    "Dystopia + residual",
]

df_2021 = df_2021.drop(columns=columns_to_drop, errors="ignore")

df_2021 = df_2021.rename(columns={
    "Logged GDP per capita": "GDP per capita",
    "Freedom to make life choices": "Life choices freedom",
})

df_panel = df_panel.rename(columns={
    "Life Ladder": "Ladder score",
    "Log GDP per capita": "GDP per capita",
    "Healthy life expectancy at birth": "Healthy life expectancy",
    "Freedom to make life choices": "Life choices freedom",
})

df_panel = df_panel.dropna(subset=["Ladder score", "year"])

display(df_2021.head(2))
display(df_panel.head(2))


,Country name,Regional indicator,Ladder score,GDP per capita,Social support,Healthy life expectancy,Life choices freedom,Generosity,Perceptions of corruption
0,Finland,Western Europe,7.842,10.775,0.954,72.0,0.949,-0.098,0.186
1,Denmark,Western Europe,7.620,10.933,0.954,72.7,0.946,0.030,0.179


,Country name,year,Ladder score,GDP per capita,Social support,Healthy life expectancy,Life choices freedom,Generosity,Perceptions of corruption,Positive affect,Negative affect
0,Afghanistan,2008,3.724,7.37,0.451,50.8,0.718,0.168,0.882,0.518,0.258
1,Afghanistan,2009,4.402,7.54,0.552,51.2,0.679,0.190,0.850,0.584,0.237


**Obeservation**:\
Detected 3 columns missing from **`df_2021`** ( `Year`, `Positive affect`, `Negative affect` ) and 1 column missing from **`df_panel`**( `Regional indicator` ). In order to be able to concat these 2 tables into 1 fact table **`fact_happiness`** to facilitate the data modeling in PowerBI, we will add up missing columns and align these 2 tables in terms of column order.


In [2]:
df_2021["Year"] = 2021
df_panel = df_panel.rename(columns= {"year":"Year"})

region_map = df_2021.set_index("Country name")["Regional indicator"]

df_panel["Regional indicator"] = df_panel["Country name"].map(region_map)

df_2021["Positive affect"] = pd.NA
df_2021["Negative affect"] = pd.NA

cols = [
    "Country name", "Year", "Regional indicator",
    "Ladder score", "GDP per capita", "Social support",
    "Healthy life expectancy", "Life choices freedom",
    "Generosity", "Perceptions of corruption",
    "Positive affect", "Negative affect"
]

fact_happiness = pd.concat([df_panel[cols],df_2021[cols]], axis = 0, ignore_index=True)

fact_happiness["Year"].head()




/tmp/ipykernel_43047/2826943815.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fact_happiness = pd.concat([df_panel[cols],df_2021[cols]], axis = 0, ignore_index=True)


,Year
0,2008
1,2009
2,2010
3,2011
4,2012


Check if country names are alligned and if "country + year“
 key is unique

In [3]:
fact_happiness.duplicated(["Country name", "Year"]).sum()

np.int64(0)

In [4]:
country_2021 = set(df_2021["Country name"])
country_panel = set(df_panel["Country name"])
print("Only in 2021:")
print(country_2021 - country_panel)

print("\nOnly in panel:")
print(country_panel - country_2021)

print(len(country_2021))
print(len(country_panel))

Only in 2021:
set()

Only in panel:
{'South Sudan', 'Angola', 'Oman', 'Sudan', 'Djibouti', 'Somalia', 'Qatar', 'Suriname', 'Congo (Kinshasa)', 'Trinidad and Tobago', 'Belize', 'Bhutan', 'Guyana', 'Cuba', 'Central African Republic', 'Syria', 'Somaliland region'}
149
166


**Observation**:\
A country consistency check was performed between the 2021 and panel datasets. The panel dataset contained additional countries that were not present in the 2021 dataset. These records were retained because they represent valid observations rather than naming inconsistencies.

However, for countries that only exist in **`df_panel`**, `Regional indicator` were not added by map() as they don't exist in **`df_2021`**. As a result, we will manually add regional indicators for these countries.



In [5]:
df_2021["Regional indicator"].unique()

array(['Western Europe', 'North America and ANZ',
       'Middle East and North Africa', 'Latin America and Caribbean',
       'Central and Eastern Europe', 'East Asia', 'Southeast Asia',
       'Commonwealth of Independent States', 'Sub-Saharan Africa',
       'South Asia'], dtype=object)

In [6]:
additional_region_map= {"Somaliland region": "Sub-Saharan Africa",
    "South Sudan": "Sub-Saharan Africa",
    "Guyana": "Latin America and Caribbean",
    "Central African Republic": "Sub-Saharan Africa",
    "Congo (Kinshasa)": "Sub-Saharan Africa",
    "Bhutan": "South Asia",
    "Djibouti": "Middle East and North Africa",
    "Qatar": "Middle East and North Africa",
    "Belize": "Latin America and Caribbean",
    "Sudan": "Sub-Saharan Africa",
    "Suriname": "Latin America and Caribbean",
    "Cuba": "Latin America and Caribbean",
    "Syria": "Middle East and North Africa",
    "Somalia": "Sub-Saharan Africa",
    "Oman": "Middle East and North Africa",
    "Trinidad and Tobago": "Latin America and Caribbean",
    "Angola": "Sub-Saharan Africa"}
fact_happiness.loc[fact_happiness["Regional indicator"].isna(),'Regional indicator'] = (
    fact_happiness.loc[fact_happiness["Regional indicator"].isna(),"Country name"]
    .map(additional_region_map))

In [7]:
print(fact_happiness["Regional indicator"].isna().sum())

0


Generate the first dimension table **`dim_country`**


In [8]:
dim_country = fact_happiness[["Country name", "Regional indicator"]].drop_duplicates().reset_index(drop = True)
dim_country.head()

,Country name,Regional indicator
0,Afghanistan,South Asia
1,Albania,Central and Eastern Europe
2,Algeria,Middle East and North Africa
3,Angola,Sub-Saharan Africa
4,Argentina,Latin America and Caribbean


Output **`fact_happiness`** and **`dim_country`** as csv.document. Will use **DAX** to create "Date table" and "Measures table"  

In [9]:
fact_happiness.to_csv("fact_happiness.csv", index=False)

dim_country.to_csv("dim_country.csv", index=False)